# AI Physics Tutor — Colab Benchmark Runner

MSc Thesis: **AI Assistant — A Multimodal Conversational Agent with Realistic Avatar for Physics Education**

Author: Kopalidis Thomas — MSc AI & DL, University of West Attica

---

## What this notebook does

1. **Setup** — Mount Google Drive, clone repo, install dependencies, download Ollama binary
2. **Prepare models** — Pull Ollama LLMs (gemma3:1b, qwen2.5:3b/7b/14b, llama3.1:8b)
3. **Build KB index** — Run `ingest_kb.py` and `build_index.py` (or restore from Drive)
4. **Smoke test** — Verify STT, RAG, LLM, TTS, Wav2Lip work end-to-end on this GPU
5. **Benchmark** (coming later) — Run the full experimental grid

## Prerequisites

- **Runtime**: GPU (T4 is enough for gemma/qwen 3b/7b; A100 needed for qwen 14b + Wav2Lip batching)
- **Account**: Google account with Drive access
- **Repo**: https://github.com/thomaskopalidis/thesis_ai_and_dl (public)

> **To change GPU type**: Runtime → Change runtime type → Hardware accelerator

## Cell 1 — Verify runtime and GPU

In [1]:
import subprocess, sys, platform
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print('')
print('=== GPU ===')
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True)
    print(out.strip())
except Exception as e:
    print(f'nvidia-smi failed: {e}  (are you on a GPU runtime?)')
print('')
print('=== Disk ===')
print(subprocess.check_output(['df', '-h', '/content'], text=True).strip())

Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39

=== GPU ===
Tesla T4, 15360 MiB, 580.82.07

=== Disk ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   71G  38% /


## Cell 2 — Mount Google Drive

Drive gives us persistent storage so Ollama model downloads and benchmark results survive runtime resets.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
THESIS_ROOT = '/content/drive/MyDrive/thesis_ai_and_dl'
os.makedirs(f'{THESIS_ROOT}/ollama_models', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/chroma_db', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/results', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/logs', exist_ok=True)
print(f'Thesis Drive folder: {THESIS_ROOT}')
!ls -la {THESIS_ROOT}

Mounted at /content/drive
Thesis Drive folder: /content/drive/MyDrive/thesis_ai_and_dl
total 24
drwx------ 6 root root 4096 Oct 10 22:11 .
drwx------ 3 root root 4096 Oct 10 22:11 ..
drwx------ 2 root root 4096 Oct 10 22:11 chroma_db
drwx------ 2 root root 4096 Oct 10 22:11 logs
drwx------ 2 root root 4096 Oct 10 22:11 ollama_models
drwx------ 2 root root 4096 Oct 10 22:11 results


## Cell 3 — Clone the project repository

Clones into `/content/thesis_ai_and_dl` (fresh each runtime). Keep **big artifacts in Drive**, keep **code in GitHub**.

In [3]:
import os, shutil
PROJECT_DIR = '/content/thesis_ai_and_dl'
REPO = 'https://github.com/thomaskopalidis/thesis_ai_and_dl.git'

if os.path.exists(PROJECT_DIR):
    print(f'Removing existing {PROJECT_DIR}')
    shutil.rmtree(PROJECT_DIR)

!git clone --depth 1 {REPO} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -5

Cloning into '/content/thesis_ai_and_dl'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 67 (delta 11), reused 66 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 1007.14 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/thesis_ai_and_dl
b8dcc26 (grafted, HEAD -> master, origin/master, origin/HEAD) feat(colab): add thesis_benchmark.ipynb for Colab/A100 execution


## Cell 4 — Install Python dependencies

We pin versions known to work together (see docs/LAB_NOTEBOOK.md for the saga).
Colab usually has torch+cuda pre-installed; we skip reinstalling torch to avoid breaking the CUDA stack.

In [8]:
# Minimal install — only what Colab lacks. No version pins (use what resolves).
import subprocess, sys

packages_to_install = [
    "faster-whisper",
    "chromadb",
    "edge-tts",
    "pymupdf",
    "soxr",
    "einops",
]

print(f'Installing: {", ".join(packages_to_install)}')
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages_to_install,
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

print('\n=== Verification ===')
import importlib
for mod in ['torch', 'numpy', 'cv2', 'librosa', 'sentence_transformers',
            'chromadb', 'fastapi', 'faster_whisper', 'edge_tts',
            'soundfile', 'scipy', 'pkg_resources', 'einops']:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '?')
        print(f'OK   {mod:28s} {v}')
    except Exception as e:
        print(f'FAIL {mod}: {type(e).__name__}: {str(e)[:100]}')

Installing: faster-whisper, chromadb, edge-tts, pymupdf, soxr, einops
 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 13.8 MB/s eta 0:00:00


=== Verification ===
OK   torch                        2.11.0+cu130
OK   numpy                        2.1.3
OK   cv2                          4.14.0
OK   librosa                      0.11.0
OK   sentence_transformers        5.7.0
OK   chromadb                     1.5.9
OK   fastapi                      0.141.1
OK   faster_whisper               1.2.1
OK   edge_tts                     7.2.8
OK   soundfile                    0.

## Cell 5 — Install Ollama binary

Ollama is downloaded as a tgz, extracted into `/content/ollama_bin`. The server will be launched in Cell 7. Models live under `$THESIS_ROOT/ollama_models` so they survive runtime resets.

In [9]:
import os, subprocess
OLLAMA_BIN_DIR = '/content/ollama_bin'
OLLAMA_URL = 'https://github.com/ollama/ollama/releases/download/v0.6.5/ollama-linux-amd64.tgz'

if not os.path.isfile(f'{OLLAMA_BIN_DIR}/bin/ollama'):
    !mkdir -p {OLLAMA_BIN_DIR}
    !wget -q -O /tmp/ollama.tgz {OLLAMA_URL}
    !tar -xzf /tmp/ollama.tgz -C {OLLAMA_BIN_DIR}
    !rm /tmp/ollama.tgz

os.environ['PATH'] = f'{OLLAMA_BIN_DIR}/bin:' + os.environ['PATH']
os.environ['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'
!ollama --version

## Cell 6 — Install ffmpeg (needed by STT and Wav2Lip output encoding)

In [10]:
!apt-get -qq update && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 >/dev/null
!ffmpeg -version | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## Cell 7 — Start Ollama server in background

Launches `ollama serve` and waits for the API to respond. We do NOT block the notebook — the server keeps running in the background.

In [16]:
import subprocess, time, requests

# Kill any previous ollama server (idempotent)
subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(1)

log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'
env = os.environ.copy()
env['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'

with open(log_path, 'ab') as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

# Wait up to 30s for server to accept requests
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'Ollama ready after {i+1}s')
            print('Available models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('[FAIL] Ollama did not start — check', log_path)

Ollama ready after 2s
Available models: []


## Cell 8 — Pull the LLM models we will benchmark

This downloads ~22 GB total **once** (into Drive). Subsequent runtimes reuse the Drive-mounted models.

If Drive is slow, consider pulling into `/content` (local SSD) and copying only final results back to Drive.

In [17]:
MODELS_TO_PULL = [
    'gemma3:1b',                       # 815 MB  — smoke test
    'qwen2.5:3b',                      # 1.9 GB
    'qwen2.5:7b',                      # 4.7 GB
    'llama3.1:8b',                     # 4.9 GB
    #'qwen2.5:14b-instruct-q4_K_M',     # 9.0 GB — only if GPU has >= 20 GB VRAM
]

for m in MODELS_TO_PULL:
    print(f'\n=== Pulling {m} ===')
    !ollama pull {m} 2>&1 | tail -3

print('\n=== Final list ===')
!ollama list


=== Pulling gemma3:1b ===

	https://ollama.com/download


=== Pulling qwen2.5:3b ===
verifying sha256 digest 
writing manifest 
success 

=== Pulling qwen2.5:7b ===
verifying sha256 digest 
writing manifest 
success 

=== Pulling llama3.1:8b ===
verifying sha256 digest 
writing manifest 
success 

=== Final list ===
NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
qwen2.5:7b     845dbda0ea48    4.7 GB    2 minutes ago             
qwen2.5:3b     357c53fb659c    1.9 GB    5 minutes ago             


In [20]:
!ollama pull gemma3:1b 2>&1 | tail -5
print('---')
!ollama list


Please download the latest version at:

	https://ollama.com/download

---
NAME           ID              SIZE      MODIFIED      
llama3.1:8b    46e0c10c039e    4.9 GB    2 minutes ago    
qwen2.5:7b     845dbda0ea48    4.7 GB    5 minutes ago    
qwen2.5:3b     357c53fb659c    1.9 GB    8 minutes ago    


## Cell 9 — Download OpenStax PDFs and build the KB index

The PDFs are not in git (too large). We download them once to Drive, run ingestion to produce `chunks.jsonl`, then build the ChromaDB index. All outputs live in Drive — next runtime restart will just reuse them.

In [24]:
import os, urllib.request

KB_DIR = f'{PROJECT_DIR}/Knowledge_base'
SRC_DIR = f'{KB_DIR}/sources'
DRIVE_PDFS = f'{THESIS_ROOT}/openstax_pdfs'
os.makedirs(DRIVE_PDFS, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

OPENSTAX = {
    'openstax_vol1.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume1-WEB_Zwi59QO.pdf',
    'openstax_vol2.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume2-WEB_oqeqFz9.pdf',
    'openstax_vol3.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume3-WEB_8rXeDQZ.pdf',
}

for fname, url in OPENSTAX.items():
    drive_path = f'{DRIVE_PDFS}/{fname}'
    project_path = f'{SRC_DIR}/{fname}'
    if not os.path.isfile(drive_path):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, drive_path)
    # Symlink from project dir into Drive copy — no duplication
    if not os.path.islink(project_path) and not os.path.isfile(project_path):
        os.symlink(drive_path, project_path)
    print(f'OK  {fname} ({os.path.getsize(drive_path)/1e6:.1f} MB)')

OK  openstax_vol1.pdf (81.5 MB)
OK  openstax_vol2.pdf (63.8 MB)
OK  openstax_vol3.pdf (53.5 MB)


In [25]:
# If a previously built chunks.jsonl exists in Drive, reuse it; otherwise re-ingest.
import os
DRIVE_CHUNKS = f'{THESIS_ROOT}/chunks.jsonl'
PROJECT_CHUNKS = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
os.makedirs(f'{PROJECT_DIR}/Knowledge_base/processed', exist_ok=True)

if os.path.isfile(DRIVE_CHUNKS):
    !cp {DRIVE_CHUNKS} {PROJECT_CHUNKS}
    print(f'Reused chunks.jsonl from Drive ({os.path.getsize(PROJECT_CHUNKS)/1e6:.1f} MB)')
else:
    !cd {PROJECT_DIR} && python evaluation/ingest_kb.py --chunk-size 220 --overlap 32 --preview 2
    !cp {PROJECT_CHUNKS} {DRIVE_CHUNKS}
    print('Fresh ingestion → saved to Drive for reuse')

config.json: 100% 612/612 [00:00<00:00, 3.21MB/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.73MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 26.4MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 54.2MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 710kB/s]
[READ] openstax_vol1.pdf
       → 116 sections kept (after exclusions)
       chunking section 1/116 (0 chunks so far)…
       chunking section 21/116 (299 chunks so far)…
       chunking section 41/116 (596 chunks so far)…
       chunking section 61/116 (872 chunks so far)…
       chunking section 81/116 (1134 chunks so far)…
       chunking section 101/116 (1412 chunks so far)…
       → 1612 chunks
[READ] openstax_vol2.pdf
       → 111 sections kept (after exclusions)
       chunking section 1/111 (0 chunks so far)…
       chunking section 21/111 (264 chunks so far)…
       chunking section 41/111 (494 chunks so far)…
       chunking section 61/111 (758 chunks so far)…
       chunking section 81/111 (966 chunks s

In [28]:
# Build (or reuse) ChromaDB index. Store the DB in Drive so it persists.
import os
DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# Symlink project's chroma_db -> Drive's chroma_db so build_index writes to Drive directly
if os.path.exists(PROJECT_CHROMA) and not os.path.islink(PROJECT_CHROMA):
    import shutil; shutil.rmtree(PROJECT_CHROMA)
if not os.path.islink(PROJECT_CHROMA):
    os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)

# Only build if the collection does not exist yet
import glob
if not glob.glob(f'{DRIVE_CHROMA}/*.sqlite3'):
    !cd {PROJECT_DIR} && python evaluation/build_index.py
else:
    print('ChromaDB index already present in Drive — skipping rebuild')
    !ls -la {DRIVE_CHROMA}

ChromaDB index already present in Drive — skipping rebuild
total 204
drwx------ 3 root root   4096 Oct 10 22:53 .
drwx------ 7 root root   4096 Oct 10 22:53 ..
drwx------ 2 root root   4096 Oct 10 22:53 bb316df4-efe7-4212-9156-2eafa8344c7a
-rw------- 1 root root 196608 Oct 10 22:53 chroma.sqlite3


In [30]:
import os, shutil, subprocess, sys

DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# 1. Remove symlink
if os.path.islink(PROJECT_CHROMA):
    os.unlink(PROJECT_CHROMA)
    print('Removed symlink')

# 2. Clean the real Drive location
if os.path.exists(DRIVE_CHROMA):
    for item in os.listdir(DRIVE_CHROMA):
        p = os.path.join(DRIVE_CHROMA, item)
        if os.path.isdir(p):
            shutil.rmtree(p)
        else:
            os.remove(p)
    print(f'Cleaned {DRIVE_CHROMA}')
else:
    os.makedirs(DRIVE_CHROMA)
    print(f'Created {DRIVE_CHROMA}')

# 3. Re-create symlink
os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)
print('Symlink re-created')

# 4. Now manually build the index WITHOUT the rmtree step
# (we already cleaned it above)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import importlib
import config
importlib.reload(config)

import json
import chromadb
from chromadb.utils import embedding_functions

CHUNKS_PATH = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
print(f'Reading chunks from {CHUNKS_PATH}')

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config.RAG_EMBEDDING_MODEL
)
client = chromadb.PersistentClient(path=DRIVE_CHROMA)

# Drop existing collection if any
try:
    client.delete_collection(config.CHROMA_COLLECTION_NAME)
    print('Dropped existing collection')
except Exception:
    pass

col = client.create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)

documents, ids, metadatas = [], [], []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        c = json.loads(line)
        documents.append(c['text'])
        ids.append(c['chunk_id'])
        metadatas.append({
            'source': c['source'],
            'volume': c['volume'],
            'chapter': c['chapter'],
            'section': c['section'],
            'page_start': c['page_start'],
            'page_end': c['page_end'],
            'license': c['license'],
            'n_tokens': c['n_tokens'],
        })

print(f'Loaded {len(documents)} chunks, indexing...')
BATCH = 256
for i in range(0, len(documents), BATCH):
    col.add(
        documents=documents[i:i+BATCH],
        ids=ids[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )
    print(f'[INDEX] {min(i+BATCH, len(documents))}/{len(documents)}')

print(f"\n[DONE] Collection '{config.CHROMA_COLLECTION_NAME}' now has {col.count()} chunks.")


Removed symlink
Cleaned /content/drive/MyDrive/thesis_ai_and_dl/chroma_db
Symlink re-created
Reading chunks from /content/thesis_ai_and_dl/Knowledge_base/processed/chunks.jsonl
Loaded 3950 chunks, indexing...
[INDEX] 256/3950
[INDEX] 512/3950
[INDEX] 768/3950
[INDEX] 1024/3950
[INDEX] 1280/3950
[INDEX] 1536/3950
[INDEX] 1792/3950
[INDEX] 2048/3950
[INDEX] 2304/3950
[INDEX] 2560/3950
[INDEX] 2816/3950
[INDEX] 3072/3950
[INDEX] 3328/3950
[INDEX] 3584/3950
[INDEX] 3840/3950
[INDEX] 3950/3950

[DONE] Collection 'physics_kb' now has 3950 chunks.


## Cell 10 — Smoke test the RAG

One retrieval query to confirm the index works and metadata is clean.

In [33]:
import sys
sys.path.insert(0, PROJECT_DIR)

import importlib
from core import rag
importlib.reload(rag)

# Direct query to see what chromadb returns
col = rag._get_collection()
print(f'Collection count: {col.count()}')
print(f'Collection name: {col.name}')

results = col.query(
    query_texts=["What is Newton's second law?"],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)
print()
print('Keys:', list(results.keys()))
print()
for k in ['ids', 'documents', 'metadatas', 'distances']:
    v = results.get(k)
    if v is None:
        print(f'{k}: None')
    elif isinstance(v, list) and v and isinstance(v[0], list):
        print(f'{k}: list of {len(v[0])} items')
        if v[0]:
            print(f'  first item type: {type(v[0][0]).__name__}')
            print(f'  first item preview: {str(v[0][0])[:120]}')
    else:
        print(f'{k}: {type(v).__name__} = {str(v)[:120]}')


[RAG] Loaded existing collection: 3 chunks
Collection count: 3
Collection name: physics_kb

Keys: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']

ids: list of 3 items
  first item type: str
  first item preview: chunk_1
documents: list of 3 items
  first item type: str
  first item preview: Newton's Second Law of Motion states that the acceleration of an object is directly proportional to the net force acting
metadatas: list of 3 items
  first item type: NoneType
  first item preview: None
distances: list of 3 items
  first item type: float
  first item preview: 0.2850174903869629


In [34]:
import os, sys

# Where is rag.py actually looking?
sys.path.insert(0, PROJECT_DIR)
import importlib
import config
importlib.reload(config)
from core import rag
importlib.reload(rag)

print('Config CHROMA_DB_DIR:', config.CHROMA_DB_DIR)
print('rag.CHROMA_DIR:', rag.CHROMA_DIR)
print('rag.KB_PATH:', rag.KB_PATH)
print('rag.COLLECTION_NAME:', rag.COLLECTION_NAME)
print()

# What's actually in both locations?
import glob
for loc in [str(config.CHROMA_DB_DIR), f'{THESIS_ROOT}/chroma_db']:
    print(f'Contents of {loc}:')
    for f in glob.glob(f'{loc}/**', recursive=True):
        print(f'  {f}')
    print()


Config CHROMA_DB_DIR: /content/thesis_ai_and_dl/chroma_db
rag.CHROMA_DIR: /content/thesis_ai_and_dl/chroma_db
rag.KB_PATH: /content/thesis_ai_and_dl/Knowledge_base/physics_kb.txt
rag.COLLECTION_NAME: physics_kb

Contents of /content/thesis_ai_and_dl/chroma_db:
  /content/thesis_ai_and_dl/chroma_db/
  /content/thesis_ai_and_dl/chroma_db/chroma.sqlite3
  /content/thesis_ai_and_dl/chroma_db/23e824a8-2a97-4a3a-abec-60013fd6a8f4
  /content/thesis_ai_and_dl/chroma_db/23e824a8-2a97-4a3a-abec-60013fd6a8f4/header.bin
  /content/thesis_ai_and_dl/chroma_db/23e824a8-2a97-4a3a-abec-60013fd6a8f4/data_level0.bin
  /content/thesis_ai_and_dl/chroma_db/23e824a8-2a97-4a3a-abec-60013fd6a8f4/length.bin
  /content/thesis_ai_and_dl/chroma_db/23e824a8-2a97-4a3a-abec-60013fd6a8f4/link_lists.bin
  /content/thesis_ai_and_dl/chroma_db/23e824a8-2a97-4a3a-abec-60013fd6a8f4/index_metadata.pickle

Contents of /content/drive/MyDrive/thesis_ai_and_dl/chroma_db:
  /content/drive/MyDrive/thesis_ai_and_dl/chroma_db/
  /co

In [37]:
import os, shutil, sys, time

DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# 1. Αφαίρεσε symlink
if os.path.islink(PROJECT_CHROMA):
    os.unlink(PROJECT_CHROMA)

# 2. Nuke completely with sleep για να αφήσει time το Drive να sync
if os.path.exists(DRIVE_CHROMA):
    shutil.rmtree(DRIVE_CHROMA)
    print('✓ Nuked Drive chroma')
time.sleep(3)

# 3. Verify διαγράφηκε
print('Still exists?', os.path.exists(DRIVE_CHROMA))

# 4. Recreate
os.makedirs(DRIVE_CHROMA)
os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)
print('✓ Fresh dirs')

# 5. Clear any Python-level chromadb caches
import importlib, sys
for mod in list(sys.modules.keys()):
    if 'chromadb' in mod or 'core.rag' in mod or mod == 'config':
        del sys.modules[mod]
print('✓ Cleared module cache')

# 6. Re-import fresh
sys.path.insert(0, PROJECT_DIR)
import config
import chromadb
from chromadb.utils import embedding_functions
import json

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config.RAG_EMBEDDING_MODEL
)
client = chromadb.PersistentClient(path=DRIVE_CHROMA)

# 7. Try delete first (idempotent)
try:
    client.delete_collection(config.CHROMA_COLLECTION_NAME)
    print(f'✓ Deleted existing collection')
except Exception as e:
    print(f'  (no existing collection to delete: {e})')

# 8. Create fresh with get_or_create as fallback
try:
    col = client.create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)
    print('✓ Created new collection')
except Exception as e:
    print(f'  create_collection failed: {e}')
    print('  Trying get_or_create...')
    col = client.get_or_create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)

# 9. Verify empty
print(f'Collection count before indexing: {col.count()}')

# 10. Load and index
CHUNKS_PATH = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
documents, ids, metadatas = [], [], []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        c = json.loads(line)
        documents.append(c['text'])
        ids.append(c['chunk_id'])
        metadatas.append({
            'source': c['source'],
            'volume': c['volume'],
            'chapter': c['chapter'],
            'section': c['section'],
            'page_start': c['page_start'],
            'page_end': c['page_end'],
            'license': c['license'],
            'n_tokens': c['n_tokens'],
        })

print(f'\n✓ Loaded {len(documents)} chunks, indexing...')
BATCH = 256
for i in range(0, len(documents), BATCH):
    col.add(
        documents=documents[i:i+BATCH],
        ids=ids[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )
    if (i // BATCH) % 4 == 0:
        print(f'[INDEX] {min(i+BATCH, len(documents))}/{len(documents)}')

print(f"\n[DONE] Collection '{config.CHROMA_COLLECTION_NAME}' now has {col.count()} chunks.")

# 11. CRITICAL verify - metadata exists
r = col.query(query_texts=["Newton second law"], n_results=1, include=["metadatas", "documents", "distances"])
print(f'\n=== VERIFY ===')
print(f'id: {r["ids"][0][0]}')
print(f'metadata: {r["metadatas"][0][0]}')


✓ Nuked Drive chroma
Still exists? False
✓ Fresh dirs
✓ Cleared module cache


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  (no existing collection to delete: Collection [physics_kb] does not exist)
✓ Created new collection
Collection count before indexing: 0

✓ Loaded 3950 chunks, indexing...
[INDEX] 256/3950
[INDEX] 1280/3950
[INDEX] 2304/3950
[INDEX] 3328/3950

[DONE] Collection 'physics_kb' now has 3950 chunks.

=== VERIFY ===
id: openstax_vol1.pdf:31:0
metadata: {'n_tokens': 188, 'page_end': 231, 'source': 'openstax_vol1.pdf', 'section': '5.5 Newton’s Third Law', 'chapter': "Chapter 5 Newton's Laws of Motion", 'volume': 'University Physics Volume 1', 'page_start': 226, 'license': 'CC BY 4.0'}


In [38]:
import sys
sys.path.insert(0, PROJECT_DIR)
from core.rag import retrieve_with_metadata

r = retrieve_with_metadata("What is Newton's second law?", top_k=3)
for i, (cid, meta, dist, doc) in enumerate(zip(r['ids'], r['metadatas'], r['distances'], r['documents']), 1):
    print(f'[{i}] dist={dist:.3f}  {meta.get("chapter")}  /  {meta.get("section")}')
    print(f'    {doc[:140]}...')
    print()

[RAG] Loaded existing collection: 3950 chunks
[1] dist=0.385  Chapter 5 Newton's Laws of Motion  /  5.5 Newton’s Third Law
    Significance
To apply Newton’s second law as the primary equation in solving a problem, we sometimes have to rely on other
equations, such a...

[2] dist=0.410  Chapter 5 Newton's Laws of Motion  /  5.5 Newton’s Third Law
    When you push on a wall, the wall pushes back on you. This brings us to Newton’s third law. Newton’s third law represents a certain symmetry...

[3] dist=0.419  Chapter 5 Newton's Laws of Motion  /  5.3 Newton's Second Law
    If the magnitude of the truck’s acceleration is
 what is the magnitude of the sports
car’s acceleration? Component Form of Newton’s Second L...



In [42]:
# Check Ollama status
import requests

try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    print('Ollama API OK')
    print('Models:', [m['name'] for m in r.json().get('models', [])])
except Exception as e:
    print(f'Ollama API DEAD: {e}')
    print('Need to restart Ollama server')


Ollama API OK
Models: ['llama3.1:8b', 'qwen2.5:7b', 'qwen2.5:3b']


In [44]:
# Get Ollama server logs
!tail -60 {THESIS_ROOT}/logs/ollama_serve.log

print('\n\n=== GPU state ===')
!nvidia-smi

print('\n=== CUDA version ===')
!nvcc --version 2>/dev/null || echo "nvcc not installed"

print('\n=== Ollama version ===')
!ollama --version


	runtime/chan.go:506 +0x12 fp=0xc00006c7b8 sp=0xc00006c790 pc=0x572b65576cd2
runtime.unique_runtime_registerUniqueMapCleanup.func2(...)
	runtime/mgc.go:1796
runtime.unique_runtime_registerUniqueMapCleanup.gowrap1()
	runtime/mgc.go:1799 +0x2f fp=0xc00006c7e0 sp=0xc00006c7b8 pc=0x572b6558912f
runtime.goexit({})
	runtime/asm_amd64.s:1700 +0x1 fp=0xc00006c7e8 sp=0xc00006c7e0 pc=0x572b655e24e1
created by unique.runtime_registerUniqueMapCleanup in goroutine 1
	runtime/mgc.go:1794 +0x85

goroutine 20 gp=0xc000107180 m=nil [GC worker (idle)]:
runtime.gopark(0x0?, 0x0?, 0x0?, 0x0?, 0x0?)
	runtime/proc.go:435 +0xce fp=0xc00006cf38 sp=0xc00006cf18 pc=0x572b655dadae
runtime.gcBgMarkWorker(0xc000103730)
	runtime/mgc.go:1423 +0xe9 fp=0xc00006cfc8 sp=0xc00006cf38 pc=0x572b65588449
runtime.gcBgMarkStartWorkers.gowrap1()
	runtime/mgc.go:1339 +0x25 fp=0xc00006cfe0 sp=0xc00006cfc8 pc=0x572b65588325
runtime.goexit({})
	runtime/asm_amd64.s:1700 +0x1 fp=0xc00006cfe8 sp=0xc00006cfe0 pc=0x572b655e24e1
created

In [46]:
import subprocess

# Install zstd (needed to extract new Ollama)
!apt-get install -y -qq zstd >/dev/null

# Kill any previous ollama
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
import time; time.sleep(2)

# Install latest Ollama
print('Installing latest Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5

!which ollama
!ollama --version


Installing latest Ollama...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [47]:
import subprocess, time, requests, os

# Kill zombie ollama processes
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(3)

# Start server with Drive-mounted models
log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'
env = os.environ.copy()
env['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'

with open(log_path, 'ab') as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'✓ Ollama ready after {i+1}s')
            print('Models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('[FAIL]')
    !tail -30 {THESIS_ROOT}/logs/ollama_serve.log


✓ Ollama ready after 5s
Models: ['llama3.1:8b', 'qwen2.5:7b', 'qwen2.5:3b']


In [49]:
import subprocess, requests

# Check if any model is loaded
print('=== Models loaded in GPU/CPU ===')
try:
    r = requests.get('http://localhost:11434/api/ps', timeout=3)
    print(r.json())
except Exception as e:
    print(f'FAIL: {e}')

print('\n=== Ollama logs (last 40) ===')
!tail -40 {THESIS_ROOT}/logs/ollama_serve.log

print('\n=== GPU state ===')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


=== Models loaded in GPU/CPU ===
{'models': [{'name': 'qwen2.5:3b', 'model': 'qwen2.5:3b', 'size': 2159374499, 'digest': '357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '3.1B', 'quantization_level': 'Q4_K_M'}, 'expires_at': '2026-10-10T23:23:20.575438923Z', 'size_vram': 2159374499, 'context_length': 4096, 'runner': 'llamacpp'}]}

=== Ollama logs (last 40) ===
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant
Hi there<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
'
srv          init: init: chat template, thinking = 0
srv          init: preserve_reasoning kwarg: not supported by template
srv  llama_server: model loaded
srv  llama_server: listening on http://127.0.0.1:34093
srv  update_slots: all slots are idle
time=2026-10-10T23:18:15.872Z level=INFO source=llama_server.go:1374 msg="llama-server started in 52.32 seconds"
time=202

In [50]:
import subprocess

# Model is already loaded in VRAM; this should respond fast
result = subprocess.run(
    ['ollama', 'run', 'qwen2.5:3b', 'Say hello.'],
    capture_output=True, text=True, timeout=180
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr[-200:] if result.stderr else '(empty)')
print('Exit:', result.returncode)


STDOUT: Hello! How can I assist you today?


STDERR: 5l⠙ 
Exit: 0


In [51]:
import os, shutil, subprocess, time, requests

# Stop ollama first
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(3)

# Copy models from Drive to local SSD
LOCAL_MODELS = '/content/ollama_models'
DRIVE_MODELS = f'{THESIS_ROOT}/ollama_models'

if not os.path.exists(LOCAL_MODELS):
    print(f'Copying {DRIVE_MODELS} → {LOCAL_MODELS}')
    print('(takes ~2-3 min for 11GB)')
    shutil.copytree(DRIVE_MODELS, LOCAL_MODELS)
print('✓ Done')
!du -sh {LOCAL_MODELS}

# Restart Ollama pointing to local models
env = os.environ.copy()
env['OLLAMA_MODELS'] = LOCAL_MODELS
log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'

with open(log_path, 'ab') as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'✓ Ollama ready after {i+1}s')
            print('Models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)


Copying /content/drive/MyDrive/thesis_ai_and_dl/ollama_models → /content/ollama_models
(takes ~2-3 min for 11GB)


KeyboardInterrupt: 

## Cell 11 — Smoke test the LLM

One question through gemma3:1b (fastest) to confirm Ollama + LLM wrapper work on GPU.

In [53]:
import time, os

MODEL = 'qwen2.5:3b'
os.environ['OLLAMA_MODEL'] = MODEL

# Re-import config and llm so they pick up the new env var
import importlib
import sys
for mod in list(sys.modules.keys()):
    if mod == 'config' or mod.startswith('core.'):
        del sys.modules[mod]

sys.path.insert(0, PROJECT_DIR)
import config
from core.llm import stream_llm_response

query = "What is Newton's second law?"
t0 = time.time()
parts = []
for token in stream_llm_response(query, model=MODEL):
    parts.append(token)
    if len(parts) == 1:
        print(f'[TIMING] first token: {time.time()-t0:.2f}s')
answer = ''.join(parts)
print(f'[TIMING] total: {time.time()-t0:.2f}s')
print(f'\nQ: {query}\nA: {answer}')


   ✗ Ollama offline: http://localhost:11434
[TIMING] first token: 0.00s
[TIMING] total: 0.00s

Q: What is Newton's second law?
A: I cannot reach my language model right now. Please make sure Ollama is running (ollama serve).


## Cell 12 — Smoke test TTS

Synthesise one short sentence to confirm edge-tts works in the sandbox.

In [ ]:
import asyncio, time
from core.tts import text_to_speech_stream

async def _test():
    t0 = time.time()
    nbytes = 0
    async for chunk in text_to_speech_stream('Hello from Google Colab.'):
        nbytes += len(chunk)
    print(f'[TIMING] TTS: {time.time()-t0:.2f}s  ({nbytes} bytes of mp3 audio)')

await _test()

## Cell 13 — Status summary

Everything above should have run without errors. If so, we are ready for the benchmark script (next step).

In [ ]:
import subprocess
print('=== Status ===')
print(f'Project:   {PROJECT_DIR}')
print(f'Drive:     {THESIS_ROOT}')
print(f'Ollama:    {OLLAMA_BIN_DIR}/bin/ollama')
print('')
print('=== Loaded models (VRAM) ===')
!ollama ps
print('')
print('=== GPU utilisation ===')
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv,noheader